In [ ]:
import re
import string
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from collections import Counter

import pandas as pd
import numpy as np
import nltk
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer
from nltk.sentiment.vader import SentimentIntensityAnalyzer

from sklearn.feature_extraction.text import TfidfVectorizer
from imblearn.over_sampling import SMOTE
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import classification_report, mean_absolute_error, mean_squared_error, r2_score
from wordcloud import WordCloud

# Descargar recursos de NLTK
nltk.download('stopwords')
nltk.download('vader_lexicon')

# Cargar archivo Parquet
ruta_archivo_parquet = r'C:\Users\gonza\OneDrive\Desktop\Proyecto Grupal Henry\archivos_ETL_finales\Maps_review_sitios.parquet'
df = pd.read_parquet(ruta_archivo_parquet)

# Filtrar texto nulo
df = df[df['text'].notnull()]

# Stopwords y stemmer
stopwords_english = set(stopwords.words('english'))
stemmer = SnowballStemmer("english")

def limpiar_texto(texto):
    texto = texto.lower()
    texto = re.sub(r'@\w+', '', texto)
    texto = re.sub(r'http\S+|www\S+|https\S+', '', texto, flags=re.MULTILINE)
    texto = texto.translate(str.maketrans('', '', string.punctuation))
    texto = re.sub(r'\d+', '', texto)
    processed_words = [stemmer.stem(word) for word in texto.split() if word not in stopwords_english]
    return ' '.join(processed_words)

# Análisis de sentimiento
analyzer = SentimentIntensityAnalyzer()
df['sentimiento_puntuacion'] = df['text'].apply(lambda x: analyzer.polarity_scores(str(x))['compound'])
df['sentimiento'] = df['sentimiento_puntuacion'].apply(lambda x: 'positivo' if x > 0 else 'negativo' if x < 0 else 'neutro')

# Limpieza de texto
df['text_limpio'] = df['text'].apply(limpiar_texto)

# Vectorización para clasificación
vectorizador = TfidfVectorizer()
X_train_tfidf = vectorizador.fit_transform(df['text_limpio'])
y_train = df['sentimiento']

# Distribución antes de SMOTE
print("Distribución inicial de clases:", Counter(y_train))
plt.figure(figsize=(8, 5))
sns.countplot(x=y_train, palette='viridis')
plt.title('Distribución de Clases Antes de SMOTE')
plt.show()

# SMOTE
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train_tfidf, y_train)

# Distribución después de SMOTE
print("Distribución después de SMOTE:", Counter(y_train_res))
plt.figure(figsize=(8, 5))
sns.countplot(x=y_train_res, palette='viridis')
plt.title('Distribución de Clases Después de SMOTE')
plt.show()

# GridSearchCV
param_grid = {
    'C': [0.01, 0.1, 1, 10, 100],
    'penalty': ['l1', 'l2'],
    'solver': ['liblinear']
}
modelo = LogisticRegression()
grid_search = GridSearchCV(modelo, param_grid, cv=5, scoring='f1_weighted', n_jobs=-1)
grid_search.fit(X_train_res, y_train_res)
best_params = grid_search.best_params_
print("Mejores parámetros:", best_params)

# Modelo final
modelo_final = LogisticRegression(**best_params)
modelo_final.fit(X_train_res, y_train_res)

# Guardar modelo y vectorizador
joblib.dump(modelo_final, 'modelo_sentimiento_v1.pkl')
joblib.dump(vectorizador, 'vectorizador_tfidf_v1.pkl')

# Evaluación
X_raw_train, X_raw_test, y_train_eval, y_test_eval = train_test_split(df['text'], df['sentimiento'], test_size=0.2, random_state=42)
X_test_cleaned = X_raw_test.apply(limpiar_texto)
X_test_tfidf = vectorizador.transform(X_test_cleaned)
y_pred = modelo_final.predict(X_test_tfidf)
print(classification_report(y_test_eval, y_pred))

# Nubes de palabras
def generar_wordcloud_por_sentimiento(sentimiento):
    textos = df[df['sentimiento'] == sentimiento]['text_limpio']
    vectorizer = TfidfVectorizer(stop_words='english', max_features=100)
    tfidf = vectorizer.fit_transform(textos)
    frecuencia = dict(zip(vectorizer.get_feature_names_out(), tfidf.sum(axis=0).A1))
    color = 'white' if sentimiento == 'positivo' else 'black'
    cmap = 'viridis' if sentimiento == 'positivo' else 'Reds'
    return WordCloud(width=800, height=400, background_color=color, colormap=cmap).generate_from_frequencies(frecuencia)

wordcloud_pos = generar_wordcloud_por_sentimiento('positivo')
wordcloud_neg = generar_wordcloud_por_sentimiento('negativo')

plt.figure(figsize=(16, 8))
plt.subplot(1, 2, 1)
plt.imshow(wordcloud_pos, interpolation='bilinear')
plt.axis('off')
plt.title('Palabras Clave - Reseñas Positivas')

plt.subplot(1, 2, 2)
plt.imshow(wordcloud_neg, interpolation='bilinear')
plt.axis('off')
plt.title('Palabras Clave - Reseñas Negativas')
plt.tight_layout()
plt.show()

# Coeficientes del modelo
tabla_palabras = pd.DataFrame({
    'Palabra': vectorizador.get_feature_names_out(),
    'Coeficiente': modelo_final.coef_[0]
})
tabla_palabras['Sentimiento'] = tabla_palabras['Coeficiente'].apply(lambda x: 'positivo' if x > 0 else 'negativo')
tabla_palabras = tabla_palabras.sort_values(by='Coeficiente', ascending=False).reset_index(drop=True)
print(tabla_palabras.head(10))

# ✅ Corrección de avg_rating basada en palabras clave
# Crear diccionario de coeficientes
coef_dict = dict(zip(tabla_palabras['Palabra'], tabla_palabras['Coeficiente']))

def score_manual_palabras(texto):
    palabras = texto.lower().split()
    return sum(coef_dict.get(palabra, 0) for palabra in palabras)

df['score_palabras'] = df['text_limpio'].apply(score_manual_palabras)

def score_a_rating(score):
    if score >= 1.0:
        return 5
    elif score >= 0.5:
        return 4
    elif score >= -0.5:
        return 3
    elif score >= -1.0:
        return 2
    else:
        return 1

df['avg_rating_corregido'] = df['score_palabras'].apply(score_a_rating)

# Visualización del score
plt.figure(figsize=(8, 5))
sns.histplot(df['score_palabras'], bins=30, kde=True)
plt.axvline(1.0, color='green', linestyle='--', label='5 estrellas')
plt.axvline(0.5, color='blue', linestyle='--', label='4 estrellas')
plt.axvline(-0.5, color='orange', linestyle='--', label='2 estrellas')
plt.axvline(-1.0, color='red', linestyle='--', label='1 estrella')
plt.title('Distribución de puntuaciones por palabras clave')
plt.xlabel('Score basado en palabras clave')
plt.ylabel('Frecuencia')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()
